# Performance comparison

In [ ]:
import geopandas as gpd
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [ ]:
fi = {}
lc = {}
perf = []
perf_lr = []
perf_rf = []

for reduction in ["fa"]:
    fi[reduction] = {}
    lc[reduction] = {}
    for model_type in ["lr", "rf"]:
        fi[reduction][model_type] = {}
        lc[reduction][model_type] = {}
        for cluster in [1, 3, 4, 5, 6, 7, 8]:
            with open(
                f"/data/uscuni-restricted/06_models/{reduction}/label_{cluster}/{model_type}/model.joblib",
                "rb",
            ) as f:
                model = joblib.load(f)
                if model_type == "rf":
                    fi[reduction][model_type][cluster] = (model.feature_importances_,)
                    perf_rf.append(
                        pd.Series(
                            {
                                "cluster": cluster,
                                "pooled_f1_macro": model.oob_f1_macro_,
                            }
                        )
                    )

                else:
                    lc[reduction][model_type][cluster] = (model.local_coef_,)
                    perf_lr.append(
                        pd.Series(
                            {
                                "cluster": cluster,
                                "oob_f1_macro": model.pooled_f1_macro_,
                            }
                        )
                    )

                perf.append(
                    pd.Series(
                        {
                            "reduction": reduction,
                            "model": model_type,
                            "cluster": cluster,
                            "accuracy": model.score_,
                            "balanced_accuracy": model.balanced_accuracy_,
                            "precision": model.precision_,
                            "recall": model.recall_,
                            "f1_macro": model.f1_macro_,
                            "f1_micro_": model.f1_micro_,
                            "f1_weighted": model.f1_weighted_,
                        }
                    )
                )
performance = pd.DataFrame(perf)
performance_lr = pd.DataFrame(perf_lr).set_index("cluster")
performance_rf = pd.DataFrame(perf_rf).set_index("cluster")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 3), sharey=True)

# First scatterplot
ax1 = sns.scatterplot(
    data=performance,
    x="cluster",
    y="balanced_accuracy",
    hue="model",
    style="reduction",
    ax=axes[0],
)
sns.despine(ax=ax1)
sns.move_legend(ax1, loc="upper left", bbox_to_anchor=(1, 1), frameon=False)
ax1.set_title("balanced_accuracy")

# Second scatterplot
ax2 = sns.scatterplot(
    data=performance,
    x="cluster",
    y="f1_macro",
    hue="model",
    style="reduction",
    ax=axes[1],
)
sns.despine(ax=ax2)
sns.move_legend(ax2, loc="upper left", bbox_to_anchor=(1, 1), frameon=False)
ax2.set_title("f1_macro")

plt.tight_layout()
plt.show()

In [ ]:
performance = performance.replace(
    {
        "pca": "PCA",
        "fa": "FA",
        "umap_dim20_nb5_euclidean": "no_dr",
        "lr": "LR",
        "rf": "RF",
        1: "Incoherent Large-Scale Homogeneous Fabric",
        2: "Incoherent Large-Scale Heterogeneous Fabric",
        3: "Incoherent Small-Scale Linear Fabric",
        4: "Incoherent Small-Scale Sparse Fabric",
        5: "Incoherent Small-Scale Compact Fabric",
        6: "Coherent Interconnected Fabric",
        7: "Coherent Dense Disjoint Fabric",
        8: "Coherent Dense Adjacent Fabric",
    }
)


performance.set_index(["reduction", "model", "cluster"])[
    "f1_macro"
].unstack().style.format("{:.4f}").background_gradient(
    cmap="GnBu", vmin=0.5, vmax=0.75, axis=0
)

In [ ]:
cluster_names = {
    1: "Incoherent Large-Scale Homogeneous Fabric",
    2: "Incoherent Large-Scale Heterogeneous Fabric",
    3: "Incoherent Small-Scale Linear Fabric",
    4: "Incoherent Small-Scale Sparse Fabric",
    5: "Incoherent Small-Scale Compact Fabric",
    6: "Coherent Interconnected Fabric",
    7: "Coherent Dense Disjoint Fabric",
    8: "Coherent Dense Adjacent Fabric",
}

results = []

for cluster in [1, 3, 4, 5, 6, 7, 8]:
    model_path = f"/data/uscuni-restricted/06_models/fa/label_{cluster}/lr/model.joblib"
    with open(model_path, "rb") as f:
        model = joblib.load(f)

    f1_values = model.local_pooled_f1_macro_

    results.append(
        {
            "cluster": cluster,
            "mean_f1_macro_lr": f1_values.mean(),
            "std_f1_macro_lr": f1_values.std(),
        }
    )

results_df = pd.DataFrame(results)

results_df["cluster"] = results_df["cluster"].replace(cluster_names)

results_df = results_df.set_index("cluster")

results_lr = results_df.transpose()
results_lr

In [ ]:
cluster_names = {
    1: "Incoherent Large-Scale Homogeneous Fabric",
    2: "Incoherent Large-Scale Heterogeneous Fabric",
    3: "Incoherent Small-Scale Linear Fabric",
    4: "Incoherent Small-Scale Sparse Fabric",
    5: "Incoherent Small-Scale Compact Fabric",
    6: "Coherent Interconnected Fabric",
    7: "Coherent Dense Disjoint Fabric",
    8: "Coherent Dense Adjacent Fabric",
}

results = []

for cluster in [1, 3, 4, 5, 6, 7, 8]:
    model_path = f"/data/uscuni-restricted/06_models/fa/label_{cluster}/rf/model.joblib"
    with open(model_path, "rb") as f:
        model = joblib.load(f)

    f1_values = model.local_oob_f1_macro_

    results.append(
        {
            "cluster": cluster,
            "mean_f1_macro_rf": f1_values.mean(),
            "std_f1_macro_rf": f1_values.std(),
        }
    )

results_df = pd.DataFrame(results)

results_df["cluster"] = results_df["cluster"].replace(cluster_names)

results_df2 = results_df.set_index("cluster")


results_rf = results_df2.transpose()
results_rf

In [ ]:
pd.concat([performance_lr, performance_rf], axis=1).transpose().style.format(
    "{:.4f}"
).background_gradient(cmap="GnBu", vmin=0.5, vmax=0.75, axis=0)

In [ ]:
a = pd.concat([results_lr, results_rf], axis=0)


a.style.format("{:.4f}").background_gradient(cmap="GnBu", vmin=0.5, vmax=0.75)

In [ ]:
a.style.format("{:.4f}").background_gradient(cmap="YlOrRd", vmin=0.035, vmax=0.22)

# Explore the importance of coefficients

# Feature importance


In [ ]:
fi_means = {}
fi_medians = {}
fi_stds = {}

for k, v in fi["fa"]["rf"].items():
    if k in [1, 2, 3, 4, 5, 6, 7, 8]:
        fi_means[k] = v[0].mean()
        fi_medians[k] = v[0].median()
        fi_stds[k] = v[0].std()

fi_means = pd.DataFrame(fi_means).rename(
    columns={
        1: "Incoherent Large-Scale Homogeneous Fabric",
        2: "Incoherent Large-Scale Heterogeneous Fabric",
        3: "Incoherent Small-Scale Linear Fabric",
        4: "Incoherent Small-Scale Sparse Fabric",
        5: "Incoherent Small-Scale Compact Fabric",
        6: "Coherent Interconnected Fabric",
        7: "Coherent Dense Disjoint Fabric",
        8: "Coherent Dense Adjacent Fabric",
    }
)
fi_medians = pd.DataFrame(fi_medians).rename(
    columns={
        1: "Incoherent Large-Scale Homogeneous Fabric",
        2: "Incoherent Large-Scale Heterogeneous Fabric",
        3: "Incoherent Small-Scale Linear Fabric",
        4: "Incoherent Small-Scale Sparse Fabric",
        5: "Incoherent Small-Scale Compact Fabric",
        6: "Coherent Interconnected Fabric",
        7: "Coherent Dense Disjoint Fabric",
        8: "Coherent Dense Adjacent Fabric",
    }
)
fi_stds = pd.DataFrame(fi_stds).rename(
    columns={
        1: "Incoherent Large-Scale Homogeneous Fabric",
        2: "Incoherent Large-Scale Heterogeneous Fabric",
        3: "Incoherent Small-Scale Linear Fabric",
        4: "Incoherent Small-Scale Sparse Fabric",
        5: "Incoherent Small-Scale Compact Fabric",
        6: "Coherent Interconnected Fabric",
        7: "Coherent Dense Disjoint Fabric",
        8: "Coherent Dense Adjacent Fabric",
    }
)

Get the variables with highest importance across all built form type

In [ ]:
# sort by largest absolute mean across models and
mask = (fi_means > fi_means.stack().quantile(0.50)).any(axis=1)
fi_means_filtered = fi_means.loc[mask]
row_abs_mean = fi_means_filtered.mean(axis=1)
fi_means_sorted = fi_means_filtered.loc[row_abs_mean.sort_values(ascending=False).index]

fi_means_sorted.style.format("{:.4f}").background_gradient(
    cmap="YlGnBu", vmin=0.02, vmax=0.07
)

Get their standard deviation

In [ ]:
fi_stds_filtered = fi_stds.loc[fi_means_sorted.index]

row_means = fi_stds_filtered.mean(axis=1)

# fa_lc_stds_sorted = fa_lc_stds_filtered.loc[row_means.sort_values(ascending=False).index]

fi_stds_filtered.style.format("{:.4f}").background_gradient(
    cmap="YlOrRd", vmin=0.01, vmax=0.04
)

Get tha variables that have both high feature importance and high standard deviation

In [ ]:
# Align
fi_means_aligned = fi_means_sorted.loc[fi_stds_filtered.index, fi_stds_filtered.columns]

# Compute ratio
fi_std_ratio = fi_stds_filtered / fi_means_aligned.abs()

row_abs_mean = fi_std_ratio.abs().mean(axis=1)
fi_ratio_sorted = fi_std_ratio.loc[row_abs_mean.sort_values(ascending=False).index]

fi_ratio_sorted.style.format("{:.2f}").background_gradient(
    cmap="RdYlGn_r", vmin=-0.2, vmax=1
)

Find out which fabrics have the highest ratio 

In [ ]:
pd.DataFrame((fi_std_ratio.abs().mean(axis=0).sort_values(ascending=False)).round(4))

Find out which variables have the highest ratio across built fabrics

In [ ]:
pd.DataFrame((fi_std_ratio.abs().mean(axis=1).sort_values(ascending=False)).round(4))

For each fabric, get the most spatially homogenous and heterogenous variables

In [ ]:
min_abs_coef = fi_means.stack().quantile(0.50)
ratio_threshold = fi_std_ratio.mean().mean()

ratios_under = []
ratios_over = []

for col in fi_means_sorted.columns:
    for var in fi_means_sorted.index:
        coef = fi_means_sorted.loc[var, col]
        std = fi_stds_filtered.loc[var, col]

        if abs(coef) <= min_abs_coef:
            continue

        ratio = std / abs(coef)

        if ratio < ratio_threshold:
            ratios_under.append(ratio)
        elif ratio > ratio_threshold:
            ratios_over.append(ratio)


quantiles_under = np.quantile(ratios_under, 0.75)
quantiles_over = np.quantile(ratios_over, 0.25)

quantiles_under, quantiles_over

In [ ]:
top_n = 10
min_abs_coef = fi_means.stack().quantile(0.50)
ratio_threshold = fi_std_ratio.mean().mean()

extremes_by_bf = {}

for col in fi_means_sorted.columns:
    records = []

    for var in fi_means_sorted.index:
        coef = fi_means_sorted.loc[var, col]
        std = fi_stds_filtered.loc[var, col]

        if abs(coef) <= min_abs_coef:
            continue

        ratio = std / abs(coef)
        records.append((var, coef, std, ratio))

    if not records:
        extremes_by_bf[col] = {"hetero": [], "homo": []}
        continue

    records_sorted = sorted(records, key=lambda x: x[3])

    most_homo = [r for r in records_sorted if r[3] < quantiles_under][:top_n]

    most_hetero = [r for r in reversed(records_sorted) if r[3] > quantiles_over][:top_n]

    extremes_by_bf[col] = {"homo": most_homo, "hetero": most_hetero}

for bf_type, groups in extremes_by_bf.items():
    print(f"\n{bf_type}")

    print("  Most homogeneous:")
    if groups["homo"]:
        for var, coef, std, ratio in groups["homo"]:
            print(f"    {var}: coef={coef:.3f}, std={std:.3f}, ratio={ratio:.3f}")
    else:
        print("    None below threshold")

    print("  Most heterogeneous:")
    if groups["hetero"]:
        for var, coef, std, ratio in groups["hetero"]:
            print(f"    {var}: coef={coef:.3f}, std={std:.3f}, ratio={ratio:.3f}")
    else:
        print("    None above threshold")